# 2. Data Cleaning

## Objective

This notebook cleans the identified data quality issues before
exploratory data analysis and machine learning.

The main cleaning tasks include:

- Handling undocumented categories in `EDUCATION`.
- Handling the undocumented category in `MARRIAGE`.
- Validating numerical features and target values.
- Preserving meaningful special values in repayment and financial variables.
- Avoiding unnecessary removal of outliers or duplicate-looking records.

Scaling, encoding, feature engineering, and class imbalance handling
will be performed in later stages.

In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv(
    "../data/raw/credit_default_raw.csv"
)

In [6]:
df.shape

(30000, 24)

In [7]:
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT_PAYMENT_NEXT_MONTH
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### Working Copy

A copy of the original dataframe is created for data cleaning.

The original dataframe is preserved so that all cleaning decisions
can be compared with the raw data when necessary.

In [9]:
clean_df = df.copy()

### EDUCATION - Cleaning

The undocumented categories `0`, `5`, and `6` were identified during
data understanding.

Since category `4` represents "Others", these undocumented categories
were grouped into category `4`.

**Action:**  
`0`, `5`, `6` → `4 (Others)`

This reduces inconsistent category definitions while preserving all records.

In [10]:
clean_df["EDUCATION"].value_counts().sort_index()

EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

In [11]:
invalid_education = clean_df["EDUCATION"].isin([0, 5, 6]).sum()

print("Rows requiring EDUCATION cleaning:", invalid_education)

Rows requiring EDUCATION cleaning: 345


In [12]:
clean_df["EDUCATION"] = clean_df["EDUCATION"].replace([0, 5, 6], 4)

In [13]:
clean_df["EDUCATION"].value_counts().sort_index()

EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64

### MARRIAGE - Cleaning

The dataset contains category `0`, which is not defined in the
dataset documentation.

Since category `3` represents "Others", category `0` was grouped
into category `3`.

**Action:**  
`0` → `3 (Others)`

No records were removed.

In [14]:
clean_df["MARRIAGE"].value_counts().sort_index()

MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

In [15]:
clean_df["MARRIAGE"] = clean_df["MARRIAGE"].replace(0, 3)

In [16]:
clean_df["MARRIAGE"].value_counts().sort_index()

MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64

### SEX - Cleaning Decision

The `SEX` feature contains only the expected categories:

- `1` = Male
- `2` = Female

No cleaning is required.

**Action:** Keep the original values.

In [17]:
clean_df["SEX"].value_counts().sort_index()

SEX
1    11888
2    18112
Name: count, dtype: int64

### AGE - Cleaning Decision

No invalid age values were identified.

The observed age range appears plausible for credit card customers.

**Action:** Keep the original values.

Potential age segmentation will be considered later during feature
engineering rather than during data cleaning.

In [18]:
clean_df["AGE"].describe()

count    30000.000000
mean        35.485500
std          9.217904
min         21.000000
25%         28.000000
50%         34.000000
75%         41.000000
max         79.000000
Name: AGE, dtype: float64

### LIMIT_BAL - Cleaning Decision

All credit limit values are positive.

Although some customers have considerably higher credit limits,
these values may represent valid high-credit customers rather than
data errors.

**Action:** Keep the original values.

Distribution and skewness will be investigated during EDA.

In [19]:
clean_df["LIMIT_BAL"].describe()

count      30000.000000
mean      167484.322667
std       129747.661567
min        10000.000000
25%        50000.000000
50%       140000.000000
75%       240000.000000
max      1000000.000000
Name: LIMIT_BAL, dtype: float64

### Repayment Status - Cleaning Decision

The repayment status variables contain special values such as
`-2`, `-1`, and `0`, in addition to positive delay values.

These values are preserved because they may represent different
repayment behaviors and may contain useful predictive information.

**Action:** Keep the original repayment status values.

No transformation is performed during data cleaning.
Their predictive behavior will be investigated during EDA and
feature engineering.

In [20]:
repayment_status_cols = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

In [21]:
for col in repayment_status_cols:
    print(f"\n{col}")
    print(clean_df[col].value_counts().sort_index())


PAY_0
PAY_0
-2     2759
-1     5686
 0    14737
 1     3688
 2     2667
 3      322
 4       76
 5       26
 6       11
 7        9
 8       19
Name: count, dtype: int64

PAY_2
PAY_2
-2     3782
-1     6050
 0    15730
 1       28
 2     3927
 3      326
 4       99
 5       25
 6       12
 7       20
 8        1
Name: count, dtype: int64

PAY_3
PAY_3
-2     4085
-1     5938
 0    15764
 1        4
 2     3819
 3      240
 4       76
 5       21
 6       23
 7       27
 8        3
Name: count, dtype: int64

PAY_4
PAY_4
-2     4348
-1     5687
 0    16455
 1        2
 2     3159
 3      180
 4       69
 5       35
 6        5
 7       58
 8        2
Name: count, dtype: int64

PAY_5
PAY_5
-2     4546
-1     5539
 0    16947
 2     2626
 3      178
 4       84
 5       17
 6        4
 7       58
 8        1
Name: count, dtype: int64

PAY_6
PAY_6
-2     4895
-1     5740
 0    16286
 2     2766
 3      184
 4       49
 5       13
 6       19
 7       46
 8        2
Name: count, dtype: int6

### Bill Amount - Cleaning Decision

Negative bill amounts are present in the dataset.

These values are not automatically considered data errors because
they may represent credit balances or customer overpayments.

**Action:** Preserve negative bill amounts.

Extreme values and distributions will be analyzed during EDA.

In [22]:
bill_cols = [
    "BILL_AMT1",
    "BILL_AMT2",
    "BILL_AMT3",
    "BILL_AMT4",
    "BILL_AMT5",
    "BILL_AMT6"
]

In [23]:
clean_df[bill_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
BILL_AMT1,30000.0,51223.330900,73635.860576,-165580.0,3558.75,22381.5,67091.00,964511.0
BILL_AMT2,30000.0,49179.075167,71173.768783,-69777.0,2984.75,21200.0,64006.25,983931.0
BILL_AMT3,30000.0,47013.154800,69349.387427,-157264.0,2666.25,20088.5,60164.75,1664089.0
BILL_AMT4,30000.0,43262.948967,64332.856134,-170000.0,2326.75,19052.0,54506.00,891586.0
BILL_AMT5,30000.0,40311.400967,60797.155770,-81334.0,1763.00,18104.5,50190.50,927171.0
BILL_AMT6,30000.0,38871.760400,59554.107537,-339603.0,1256.00,17071.0,49198.25,961664.0


In [24]:
(clean_df[bill_cols] < 0).sum()

BILL_AMT1    590
BILL_AMT2    669
BILL_AMT3    655
BILL_AMT4    675
BILL_AMT5    655
BILL_AMT6    688
dtype: int64

### Payment Amount - Cleaning Decision

No negative payment amounts were identified.

Zero payment amounts are preserved because a value of `0` may indicate
that the customer did not make a payment during that period.

This represents meaningful customer behavior rather than missing data.

**Action:** Keep all payment amount values.

In [25]:
payment_cols = [
    "PAY_AMT1",
    "PAY_AMT2",
    "PAY_AMT3",
    "PAY_AMT4",
    "PAY_AMT5",
    "PAY_AMT6"
]

In [26]:
clean_df[payment_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
PAY_AMT1,30000.0,5663.580500,16563.280354,0.0,1000.00,2100.0,5006.00,873552.0
PAY_AMT2,30000.0,5921.163500,23040.870402,0.0,833.00,2009.0,5000.00,1684259.0
PAY_AMT3,30000.0,5225.681500,17606.961470,0.0,390.00,1800.0,4505.00,896040.0
PAY_AMT4,30000.0,4826.076867,15666.159744,0.0,296.00,1500.0,4013.25,621000.0
PAY_AMT5,30000.0,4799.387633,15278.305679,0.0,252.50,1500.0,4031.50,426529.0
PAY_AMT6,30000.0,5215.502567,17777.465775,0.0,117.75,1500.0,4000.00,528666.0


In [27]:
(clean_df[payment_cols] < 0).sum()

PAY_AMT1    0
PAY_AMT2    0
PAY_AMT3    0
PAY_AMT4    0
PAY_AMT5    0
PAY_AMT6    0
dtype: int64

### Duplicate Records

Rows with identical feature values are not removed automatically.

Because the customer identifier is not included in the modeling
dataframe, identical feature values do not necessarily indicate that
the records belong to the same customer.

**Action:** Preserve the records unless true duplicate customer
records can be confirmed.

In [28]:
clean_df.duplicated().sum()

np.int64(35)

### Target Validation

The target variable contains only the expected binary classes:

- `0` = No default
- `1` = Default

**Action:** No cleaning is required for the target variable.

In [30]:
clean_df["DEFAULT_PAYMENT_NEXT_MONTH"].value_counts().sort_index()

DEFAULT_PAYMENT_NEXT_MONTH
0    23364
1     6636
Name: count, dtype: int64

## Data Cleaning Summary

The data cleaning process focused on correcting documented data quality
issues while preserving potentially meaningful customer behavior.

### Changes Applied

- Undocumented `EDUCATION` categories `0`, `5`, and `6` were grouped into
  category `4` (`Others`).
- Undocumented `MARRIAGE` category `0` was grouped into category `3`
  (`Others`).
- No missing-value imputation was required.
- Repayment status values were preserved without modification.
- Negative bill amounts were preserved because they may represent valid
  account balances.
- Zero payment amounts were preserved because they represent meaningful
  repayment behavior.
- No automatic outlier removal was performed.
- Duplicate-looking records were not automatically removed.
- The target variable was confirmed to contain valid binary classes.

The cleaned dataset is now ready for exploratory data analysis.

In [32]:
assert clean_df.isnull().sum().sum() == 0

assert set(clean_df["SEX"].unique()).issubset({1, 2})

assert set(clean_df["EDUCATION"].unique()).issubset({
    1, 2, 3, 4
})

assert set(clean_df["MARRIAGE"].unique()).issubset({
    1, 2, 3
})

assert (clean_df["AGE"] > 0).all()

assert (clean_df["LIMIT_BAL"] > 0).all()

assert set(clean_df["DEFAULT_PAYMENT_NEXT_MONTH"].unique()).issubset({
    0, 1
})

assert (clean_df[payment_cols] >= 0).all().all()

print("All data quality checks passed.")

All data quality checks passed.


In [33]:
cleaning_summary = pd.DataFrame({
    "Feature": [
        "EDUCATION",
        "MARRIAGE",
        "SEX",
        "AGE",
        "LIMIT_BAL",
        "PAY_*",
        "BILL_AMT*",
        "PAY_AMT*",
        "DEFAULT"
    ],

    "Action": [
        "Cleaned",
        "Cleaned",
        "Kept",
        "Kept",
        "Kept",
        "Kept",
        "Kept",
        "Kept",
        "Kept"
    ],

    "Details": [
        "0, 5, 6 merged into category 4 (Others)",
        "0 merged into category 3 (Others)",
        "Valid categories",
        "No invalid age values",
        "Positive credit limits",
        "Special repayment codes preserved",
        "Negative values preserved",
        "Zero payments preserved",
        "Valid binary target"
    ]
})

cleaning_summary

,Feature,Action,Details
0,EDUCATION,Cleaned,"0, 5, 6 merged into category 4 (Others)"
1,MARRIAGE,Cleaned,0 merged into category 3 (Others)
2,SEX,Kept,Valid categories
3,AGE,Kept,No invalid age values
4,LIMIT_BAL,Kept,Positive credit limits
5,PAY_*,Kept,Special repayment codes preserved
6,BILL_AMT*,Kept,Negative values preserved
7,PAY_AMT*,Kept,Zero payments preserved
8,DEFAULT,Kept,Valid binary target


In [34]:
clean_df.to_csv(
    "../data/processed/credit_default_clean.csv",
    index=False
)

In [35]:
print(clean_df.shape)
clean_df.head()

(30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT_PAYMENT_NEXT_MONTH
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0
